# Notebook 1:

In [ ]:
import io, time
import requests
import numpy as np
import pandas as pd

BASE = "https://data.cityofnewyork.us/resource/h9gi-nx95.csv"

FEATURE_START, FEATURE_END = "2021-01-01", "2023-01-01"   # period used to build features
TARGET_START,  TARGET_END  = "2023-01-01", "2025-01-01"   # period used to build the target
GRID_DEG = 0.02                                           # spatial block size in degrees (about 2 km)

# What counts as one crossing:
#   "street_pair" = pair of street names (on_street + cross_street)
#   "grid"        = small map cell (CELL_DEG degrees, about 50 m), used if street names are missing
UNIT = "grid"
CELL_DEG = 0.001

COLS = ["collision_id", "crash_date", "borough", "latitude", "longitude",
        "on_street_name", "cross_street_name",
        "number_of_pedestrians_injured", "number_of_pedestrians_killed"]

In [ ]:
def fetch(start, end, page=50000):
    # Download crashes between start (inclusive) and end (exclusive), page by page.
    frames, offset = [], 0
    while True:
        params = {
            "$select": ",".join(COLS),
            "$where": "crash_date >= '" + start + "T00:00:00' AND crash_date < '" + end + "T00:00:00'",
            "$order": "collision_id",
            "$limit": page,
            "$offset": offset,
        }
        r = requests.get(BASE, params=params, timeout=180)
        if r.status_code != 200:
            print("Request failed:", r.status_code)
            print(r.text[:500])
            raise SystemExit
        chunk = pd.read_csv(io.StringIO(r.text))
        if chunk.empty:
            break
        frames.append(chunk)
        offset += page
        print(start, "->", end, "rows so far:", sum(len(f) for f in frames))
        if len(chunk) < page:
            break
    return pd.concat(frames, ignore_index=True)

df_f = fetch(FEATURE_START, FEATURE_END)
df_f["period"] = "feature"
df_t = fetch(TARGET_START, TARGET_END)
df_t["period"] = "target"
raw = pd.concat([df_f, df_t], ignore_index=True)
print("Loaded rows:", len(raw))
print(raw.columns.tolist())

2021-01-01 -> 2023-01-01 rows so far: 50000
2021-01-01 -> 2023-01-01 rows so far: 100000
2021-01-01 -> 2023-01-01 rows so far: 150000
2021-01-01 -> 2023-01-01 rows so far: 200000
2021-01-01 -> 2023-01-01 rows so far: 214445
2023-01-01 -> 2025-01-01 rows so far: 50000
2023-01-01 -> 2025-01-01 rows so far: 100000
2023-01-01 -> 2025-01-01 rows so far: 150000
2023-01-01 -> 2025-01-01 rows so far: 187923
Loaded rows: 402368
['collision_id', 'crash_date', 'borough', 'latitude', 'longitude', 'on_street_name', 'cross_street_name', 'number_of_pedestrians_injured', 'number_of_pedestrians_killed', 'period']


In [ ]:
def norm_street(s):
    s = s.fillna("").astype(str).str.upper().str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    for a, b in [(" AVENUE", " AVE"), (" STREET", " ST"), (" BOULEVARD", " BLVD"), (" ROAD", " RD")]:
        s = s.str.replace(a, b, regex=False)
    return s

# ---- diagnostics: what did we actually download? ----
print("raw shape:", raw.shape)
print("Share of non-empty values per column:")
print(raw.notna().mean().round(3))
print(raw.head(3).T)

df = raw.copy()
df["on_s"] = norm_street(df["on_street_name"])
df["cross_s"] = norm_street(df["cross_street_name"])
df["ped_harm"] = df["number_of_pedestrians_injured"].fillna(0) + df["number_of_pedestrians_killed"].fillna(0)
n_loaded = len(df)

# step 1: valid coordinates inside the NYC bounding box
ok_coord = df["latitude"].between(40.4, 41.0) & df["longitude"].between(-74.3, -73.6)
n_bad_coord = int((~ok_coord).sum())
df = df[ok_coord].copy()

# step 2: define the crossing
if UNIT == "street_pair":
    has_streets = (df["on_s"] != "") & (df["cross_s"] != "")
    n_no_streets = int((~has_streets).sum())
    df = df[has_streets].copy()
    a, b = df["on_s"], df["cross_s"]
    df["ikey"] = np.where(a < b, a + "|" + b, b + "|" + a)
else:
    n_no_streets = 0
    df["ikey"] = (np.floor(df["latitude"] / CELL_DEG).astype(int).astype(str) + "_" +
                  np.floor(df["longitude"] / CELL_DEG).astype(int).astype(str))

print()
print("Unit of analysis:                   ", UNIT)
print("Rows loaded:                        ", n_loaded)
print("Removed, missing/invalid coords:    ", n_bad_coord)
print("Removed, no cross street:           ", n_no_streets, round(100 * n_no_streets / max(n_loaded, 1), 1), "%")
print("Rows kept:                          ", len(df))

if len(df) == 0:
    print("No rows left after filtering, check the numbers above.")

raw shape: (402368, 10)
Share of non-empty values per column:
collision_id                     1.000
crash_date                       1.000
borough                          0.676
latitude                         0.920
longitude                        0.920
on_street_name                   0.725
cross_street_name                0.275
number_of_pedestrians_injured    1.000
number_of_pedestrians_killed     1.000
period                           1.000
dtype: float64
                                                              0  \
collision_id                                            4073803   
crash_date                              2021-02-03T00:00:00.000   
borough                                                   BRONX   
latitude                                              40.877705   
longitude                                             -73.90566   
on_street_name                 BROADWAY                           
cross_street_name                                           NaN 

In [ ]:
feat = (df[df["period"] == "feature"].groupby("ikey").agg(
            hist_crashes=("collision_id", "count"),
            hist_ped_harm=("ped_harm", "sum"),
            lat=("latitude", "mean"),
            lon=("longitude", "mean"),
            borough=("borough", lambda s: s.dropna().mode().iat[0] if s.notna().any() else "UNKNOWN")))

targ = (df[df["period"] == "target"].groupby("ikey").agg(
            target_ped_harm=("ped_harm", "sum"),
            target_crashes=("collision_id", "count")))

# candidate crossings = intersections seen in the feature period
inter = feat.join(targ, how="left").fillna({"target_ped_harm": 0, "target_crashes": 0})
inter["y"] = (inter["target_ped_harm"] > 0).astype(int)

# spatial block id (used later for spatial cross-validation)
inter["block"] = (np.floor(inter["lat"] / GRID_DEG).astype(int).astype(str) + "_" +
                  np.floor(inter["lon"] / GRID_DEG).astype(int).astype(str))
inter.head()

,hist_crashes,hist_ped_harm,lat,lon,borough,target_ped_harm,target_crashes,y,block
ikey,,,,,,,,,
40499_-74239,1,0,40.499710,-74.238920,STATEN ISLAND,0.0,1.0,0,2024_-3712
40500_-74245,1,0,40.500896,-74.244110,STATEN ISLAND,0.0,0.0,0,2025_-3713
40501_-74242,1,0,40.501000,-74.241280,STATEN ISLAND,0.0,0.0,0,2025_-3713
40501_-74244,1,0,40.501800,-74.243473,UNKNOWN,0.0,1.0,0,2025_-3713
40502_-74236,1,0,40.502293,-74.235054,STATEN ISLAND,0.0,0.0,0,2025_-3712


In [ ]:
print("Intersections (candidate crossings):", len(inter))
print("Positive rate (target period has pedestrian harm):", round(100 * inter["y"].mean(), 2), "%")
print()
print("Target (pedestrians injured/killed per intersection):")
print(inter["target_ped_harm"].describe())
print()
print("Spearman correlation, past vs future pedestrian harm:")
print(inter[["hist_ped_harm", "target_ped_harm"]].corr(method="spearman"))
print()
per_block = inter.groupby("block").size()
print("Spatial blocks:", len(per_block),
      "| intersections per block: min", per_block.min(),
      "median", int(per_block.median()), "max", per_block.max())
print()
print("Top 10 intersections by target-period pedestrian harm:")
print(inter.sort_values("target_ped_harm", ascending=False).head(10)[["borough", "hist_ped_harm", "target_ped_harm"]])

inter.to_csv("intersections.csv")
try:
    from google.colab import files
    files.download("intersections.csv")
except Exception:
    pass

Intersections (candidate crossings): 36679
Positive rate (target period has pedestrian harm): 25.51 %

Target (pedestrians injured/killed per intersection):
count    36679.000000
mean         0.467734
std          1.091405
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max         21.000000
Name: target_ped_harm, dtype: float64

Spearman correlation, past vs future pedestrian harm:
                 hist_ped_harm  target_ped_harm
hist_ped_harm         1.000000         0.393823
target_ped_harm       0.393823         1.000000

Spatial blocks: 265 | intersections per block: min 1 median 132 max 362

Top 10 intersections by target-period pedestrian harm:
                borough  hist_ped_harm  target_ped_harm
ikey                                                   
40867_-73924  MANHATTAN              2             21.0
40631_-73947   BROOKLYN              9             17.0
40757_-73990  MANHATTAN              5             15.0
40718_-73989  MANHATT

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>